In [2]:
# import common libraires
import os
from pathlib import Path
from typing import TypedDict, Annotated, List, Optional

# import llm, parser, ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

# import langgraph libraries
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv

# Rag libraries
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

load_dotenv()

c:\Users\Arbaz Khan\Desktop\Langgraph\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [3]:
docs = (
    PyPDFLoader("./Documents/book3.pdf").load()
)

In [4]:
chunks = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=150).split_documents(docs)
for d in chunks:
    d.page_content = d.page_content.encode("utf-8", "ignore").decode("utf-8", "ignore")

len(chunks)

1652

In [6]:
encodding = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(chunks, encodding)

In [7]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})
retriever.invoke("Gradient Descent")

[Document(id='e654c62c-b3ff-4044-afb8-fa50a6f2858d', metadata={'producer': 'Antenna House PDF Output Library 6.2.609 (Linux64)', 'creator': 'AH CSS Formatter V6.2 MR4 for Linux64 : 6.2.6.18551 (2014/09/24 15:00JST)', 'creationdate': '2017-03-10T21:55:34+00:00', 'author': 'Aurélien Géron', 'moddate': '2017-07-29T21:43:02+08:00', 'title': 'Hands-On Machine Learning with Scikit-Learn and TensorFlow', 'trapped': '/False', 'source': './Documents/book3.pdf', 'total_pages': 564, 'page': 6, 'page_label': 'v'}, page_content='Batch Gradient Descent                                                                                           114\nStochastic Gradient Descent                                                                                   117\nMini-batch Gradient Descent                                                                                 119\nPolynomial Regression                                                                                                121\nLearning C

In [8]:
llm = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct", temperature=0.6)
llm.invoke("What is Gradient Descent?").content

"**Gradient Descent**\n=====================\n\nGradient Descent is a popular optimization algorithm used in machine learning and deep learning to minimize the loss function of a model. It's an iterative method that adjusts the model's parameters to reduce the error between predicted and actual outputs.\n\n**How Gradient Descent Works**\n-----------------------------\n\nThe goal of Gradient Descent is to find the optimal values of the model's parameters that result in the lowest loss. The process involves the following steps:\n\n1. **Initialize Parameters**: Initialize the model's parameters (weights and biases) randomly or with pre-trained values.\n2. **Compute Loss**: Calculate the loss between the predicted output and the actual output using a loss function (e.g., Mean Squared Error or Cross-Entropy).\n3. **Compute Gradient**: Calculate the gradient of the loss function with respect to each parameter. The gradient represents the direction of the steepest ascent.\n4. **Update Paramet

### Langgraph Orchastrator

In [9]:
min_score = 0.3
max_score = 0.7

In [12]:
class CRagState(TypedDict):
    question: str
    docs: List[Document]
    good_docs: List[Document]
    final_doc: Document

    choose_path: str
    
    answer: str

In [13]:
def RetrieveNode(state: CRagState):
    q = state["question"]
    return {"docs": retriever.invoke(q)}

In [ ]:
eval_prompt = ChatPromptTemplate(
    [
        ("system", "{document}"),
        ("human", "")
    ]
)
def EvaluatorNode(state: CRagState):
    eval_score = []
    good_docs = []
    is_good = True
    for d in state["docs"]:
        score = (eval_prompt | llm).invoke({"document":d.page_content}).content
        if float(score) < min_score: # type: ignore
            continue
        elif float(score) < max_score: # type: ignore
            is_good = False
             
        eval_score.append(score)
        good_docs.append(d)
    
    if eval_score == []:
        return {"choose_path": "knowledge_search_node"}
    elif is_good:
        return {"choose_path": "knowledge_refinement_node", "good_docs": good_docs}
    else:
        return {"choose_path": "both_node", "good_docs": good_docs}

it is none
